In [1]:
import pandas as pd

dfT1 = pd.read_csv("../data/Training dataset 1.csv")
dfT2 = pd.read_csv("../data/Training dataset 2.csv")

dfT1.columns = dfT1.columns.str.strip()
dfT2.columns = dfT2.columns.str.strip()

dfT1["DATETIME"] = pd.to_datetime(dfT1["DATETIME"], format="%d/%m/%y %H")
dfT2["DATETIME"] = pd.to_datetime(dfT2["DATETIME"], format="%d/%m/%y %H")

In [2]:
cont_cols = [c for c in dfT1.columns if c.startswith(("L_", "F_", "P_"))]
onoff_cols = [c for c in dfT1.columns if c.startswith("S_")]

print(len(cont_cols), len(onoff_cols))
print(cont_cols)
print(onoff_cols)

leftover = [c for c in dfT1.columns if c not in cont_cols + onoff_cols]
print(leftover)

31 12
['L_T1', 'L_T2', 'L_T3', 'L_T4', 'L_T5', 'L_T6', 'L_T7', 'F_PU1', 'F_PU2', 'F_PU3', 'F_PU4', 'F_PU5', 'F_PU6', 'F_PU7', 'F_PU8', 'F_PU9', 'F_PU10', 'F_PU11', 'F_V2', 'P_J280', 'P_J269', 'P_J300', 'P_J256', 'P_J289', 'P_J415', 'P_J302', 'P_J306', 'P_J307', 'P_J317', 'P_J14', 'P_J422']
['S_PU1', 'S_PU2', 'S_PU3', 'S_PU4', 'S_PU5', 'S_PU6', 'S_PU7', 'S_PU8', 'S_PU9', 'S_PU10', 'S_PU11', 'S_V2']
['DATETIME', 'ATT_FLAG']


In [3]:
n = len(dfT1)
gap = 24

cut1 = int(n * 0.6)
cut2 = int(n * 0.8)

train = dfT1.iloc[:cut1]
val = dfT1.iloc[cut1 + gap:cut2]
cal = dfT1.iloc[cut2 + gap:]

for name, block in [("Training", train), ("Validation", val), ("Calibration", cal)]:
    print(name, len(block), "rows:", block["DATETIME"].min(), "to", block["DATETIME"].max())

print("Total used:", len(train) + len(val) + len(cal), "of", n)

Training 5256 rows: 2014-01-06 00:00:00 to 2014-08-12 23:00:00
Validation 1728 rows: 2014-08-14 00:00:00 to 2014-10-24 23:00:00
Calibration 1729 rows: 2014-10-26 00:00:00 to 2015-01-06 00:00:00
Total used: 8713 of 8761


In [4]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(train[cont_cols])

train_scaled = train.copy()
train_scaled[cont_cols] = scaler.transform(train[cont_cols])

check = train_scaled[cont_cols].agg(["min", "max"]).T
print(check.round(3))

        min  max
L_T1    0.0  1.0
L_T2    0.0  1.0
L_T3    0.0  1.0
L_T4    0.0  1.0
L_T5    0.0  1.0
L_T6    0.0  1.0
L_T7    0.0  1.0
F_PU1   0.0  1.0
F_PU2   0.0  1.0
F_PU3   0.0  0.0
F_PU4   0.0  1.0
F_PU5   0.0  0.0
F_PU6   0.0  1.0
F_PU7   0.0  1.0
F_PU8   0.0  1.0
F_PU9   0.0  0.0
F_PU10  0.0  1.0
F_PU11  0.0  1.0
F_V2    0.0  1.0
P_J280  0.0  1.0
P_J269  0.0  1.0
P_J300  0.0  1.0
P_J256  0.0  1.0
P_J289  0.0  1.0
P_J415  0.0  1.0
P_J302  0.0  1.0
P_J306  0.0  1.0
P_J307  0.0  1.0
P_J317  0.0  1.0
P_J14   0.0  1.0
P_J422  0.0  1.0


In [8]:
val_scaled = val.copy()
val_scaled[cont_cols] = scaler.transform(val[cont_cols])

cal_scaled = cal.copy()
cal_scaled[cont_cols] = scaler.transform(cal[cont_cols])

dfT2_scaled = dfT2.copy()
dfT2_scaled[cont_cols] = scaler.transform(dfT2[cont_cols])

check2 = val_scaled[cont_cols].agg(["min", "max"]).T
print(check2.round(3))

check3 = cal_scaled[cont_cols].agg(["min", "max"]).T
print(check3.round(3))

check4 = dfT2_scaled[cont_cols].agg(["min", "max"]).T
print(check4.round(3))

for name, block in [("Training", train_scaled), ("Validation", val_scaled),
                    ("Calibration", cal_scaled), ("Dataset 2", dfT2_scaled)]:
    print(name, block[onoff_cols].isin([0, 1]).all().all())

          min    max
L_T1    0.019  1.010
L_T2   -0.001  0.996
L_T3    0.008  0.993
L_T4    0.032  0.991
L_T5   -0.002  1.001
L_T6    0.047  1.000
L_T7    0.036  1.000
F_PU1  -0.011  0.995
F_PU2   0.000  0.996
F_PU3   0.000  0.000
F_PU4   0.000  0.988
F_PU5   0.000  0.000
F_PU6   0.000  0.000
F_PU7   0.000  1.001
F_PU8   0.000  0.993
F_PU9   0.000  0.000
F_PU10  0.000  1.004
F_PU11  0.000  0.000
F_V2    0.000  1.019
P_J280  0.012  0.998
P_J269  0.005  1.011
P_J300  0.002  0.986
P_J256  0.007  1.014
P_J289  0.003  0.986
P_J415 -0.015  0.723
P_J302  0.097  1.018
P_J306  0.027  1.008
P_J307  0.103  1.018
P_J317  0.011  0.649
P_J14   0.006  1.020
P_J422  0.002  0.984
          min    max
L_T1    0.034  0.980
L_T2   -0.000  1.000
L_T3    0.008  0.996
L_T4    0.018  0.990
L_T5   -0.001  0.986
L_T6    0.023  1.000
L_T7    0.061  0.999
F_PU1  -0.003  0.995
F_PU2   0.000  0.996
F_PU3   0.000  0.000
F_PU4   0.000  1.006
F_PU5   0.000  0.000
F_PU6   0.000  0.974
F_PU7   0.000  0.998
F_PU8   0.000

In [9]:
import os
import joblib

out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

train_scaled.to_csv(f"{out_dir}/train_scaled.csv", index=False)
val_scaled.to_csv(f"{out_dir}/val_scaled.csv", index=False)
cal_scaled.to_csv(f"{out_dir}/cal_scaled.csv", index=False)
dfT2_scaled.to_csv(f"{out_dir}/dataset2_scaled.csv", index=False)

joblib.dump(scaler, f"{out_dir}/scaler.joblib")

test_load = pd.read_csv(f"{out_dir}/train_scaled.csv", parse_dates=["DATETIME"])
print(test_load.shape, train_scaled.shape)
print(os.listdir(out_dir))

(5256, 45) (5256, 45)
['cal_scaled.csv', 'dataset2_scaled.csv', 'scaler.joblib', 'train_scaled.csv', 'val_scaled.csv']
